# Phase B — TPU: HBO vs Random (Fixed Design)
## Validates J_topo screening with proper experimental controls

**Changes from naive design:**
1. GP features: norm one-hot + width + depth + skip + J_topo + L1_loss
2. LN J_topo NOT zeroed (conv-layer-only)
3. L1 fidelity: 5 → **10 epochs**
4. Stratified sampling (equal norm/skip representation)
5. Random arm: 30 configs
6. **LN cooling validation** first to fill beta(gamma) gap

## Step 1: TPU Setup

In [ ]:

import os, torch
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
try:
    DEVICE = xm.xla_device()
    _ = torch.zeros((2,3), device=DEVICE)
    print("TPU:", DEVICE)
except:
    raise RuntimeError("TPU not accessible. Set Accelerator to TPU.")



## Step 2: Clone

In [ ]:

get_ipython().system("cd /kaggle/working && rm -rf ThermoRG-NN")
from kaggle_secrets import UserSecretsClient
gh = UserSecretsClient().get_secret("gh_token")
get_ipython().system(f"git clone https://{gh}@github.com/xliu203/ThermoRG-NN.git --branch develop")
get_ipython().system("cd ThermoRG-NN && git config --global user.email xliu203@asu.edu && git config --global user.name Leo")
import sys
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
print("Cloned.")



## Step 3: Imports

In [ ]:

import sys, math, time, json, random, warnings, os
import numpy as np
from pathlib import Path
import torch, torch.nn as nn
from scipy.optimize import curve_fit
from scipy.linalg import svd
from scipy.stats import spearmanr
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
from thermorg import compute_J_topo
print("Imports OK")



## Step 4: Network + J_topo (LN-aware)

In [ ]:

class ConvBlock(nn.Module):
    def __init__(self, ci, co, norm='none', skip=False, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(ci, co, 3, padding=1, stride=stride, bias=False)
        if norm == 'batchnorm': self.norm = nn.BatchNorm2d(co)
        elif norm == 'layernorm': self.norm = nn.LayerNorm([co, 32//stride, 32//stride])
        else: self.norm = nn.Identity()
        self.act = nn.GELU()
        self.skip = skip and stride == 1 and ci == co

    def forward(self, x):
        out = self.act(self.norm(self.conv(x)))
        if self.skip: out = out + x
        return out

class ThermoNet(nn.Module):
    def __init__(self, base_ch=64, depth=3, norm='none', skip=False):
        super().__init__()
        ch = [3] + [base_ch]*depth
        self.blocks = nn.ModuleList([
            ConvBlock(ch[i], ch[i+1], norm, skip and i > 0)
            for i in range(depth)
        ])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(ch[-1], 10)

    def forward(self, x):
        for b in self.blocks: x = b(x)
        return self.fc(self.pool(x).squeeze())

    def get_conv_weights(self):
        # All conv layers (LN uses its own norm, but still contributes conv weights)
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]



In [ ]:

def get_J_topo(base_ch, depth, norm, skip):
    model = ThermoNet(base_ch=base_ch, depth=depth, norm=norm, skip=skip)
    J, _ = compute_J_topo(model)
    del model
    return J

print("J_topo: LN uses conv layers only (NOT zeroed)")



## Step 5: DataLoaders

In [ ]:

BATCH = 256
LR = 0.01; WD = 5e-4; MOM = 0.9
OUT = Path("/kaggle/working/phase_b_fixed"); OUT.mkdir(exist_ok=True)
CKPT = OUT/"ckpt"; CKPT.mkdir(exist_ok=True)

tfm = T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),
                 T.ToTensor(),T.Normalize([.4914,.4822,.4465],[.2470,.2435,.2616])])
tvf = T.Compose([T.ToTensor(),T.Normalize([.4914,.4822,.4465],[.2470,.2435,.2616])])
train_ds = CIFAR10("./data", True, tfm, download=True)
val_ds   = CIFAR10("./data", False, tvf, download=True)
def loaders(b=BATCH):
    return (DataLoader(train_ds, batch_size=b, shuffle=True, num_workers=0), DataLoader(val_ds, batch_size=b, shuffle=False, num_workers=0))
print(f"Data: {len(train_ds)} train, {len(val_ds)} val")



## Step 6: Training Function

In [ ]:
def train(model, tr, va, ep, lr=LR, wd=WD, mom=MOM):
    model = model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=mom, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep)
    crit = nn.CrossEntropyLoss()
    best_l, best_a, best_e = float('inf'), 0.0, 0
    for e in range(ep):
        model.train()
        for X, y in pl.MpDeviceLoader(tr, DEVICE):
            opt.zero_grad()
            loss = crit(model(X), y)
            loss.backward()
            xm.optimizer_step(opt, barrier=True)
        sch.step()
        model.eval()
        # FIX: Accumulate on TPU tensors, .item() only once per epoch
        ls_t = torch.tensor(0.0, device=DEVICE)
        ca_t = torch.tensor(0.0, device=DEVICE)
        to_t = torch.tensor(0.0, device=DEVICE)
        with torch.no_grad():
            for X, y in pl.MpDeviceLoader(va, DEVICE):
                o = model(X)
                ls_t += crit(o, y) * X.size(0)
                ca_t += (o.argmax(1)==y).type(torch.float32).sum()
                to_t += X.size(0)
        vl = (ls_t / to_t).item()
        acc = (ca_t / to_t).item()
        if vl < best_l:
            best_l, best_a, best_e = vl, acc, e+1
    del model
    return {"best_val_loss": best_l, "best_val_acc": best_a, "best_epoch": best_e}


## ══ Part 1: LN Cooling Validation ══
**Goal:** Measure gamma_LN and beta_LN to fill beta(gamma) gap for LayerNorm
**Design:** 4 D values x 2 seeds = 8 runs x 200 epochs
**Expected:** gamma_LN between None(3.39) and BN(2.29)

In [ ]:
# D_val = model width (base_ch), not dataset size
# CIFAR-10 has 50K train images — dataset subsampling is not needed for width scaling
LN_DVALS = [32, 48, 64, 96]   # model widths for scaling law
LN_SEEDS = [42, 123]
LN_CFGS  = [(d,s) for d in LN_DVALS for s in LN_SEEDS]
print(f"LN validation: {len(LN_CFGS)} runs (D = model width: {LN_DVALS})")
J_LN = get_J_topo(64, 3, 'layernorm', False)  # J_topo reference (same depth)
print(f"LN J_topo (ref): {J_LN:.4f}")


In [ ]:
ln_results = []
t0 = time.time()
for i, (D_val, seed) in enumerate(LN_CFGS):
    jf = OUT / f"ln_D{D_val}_s{seed}.json"
    if jf.exists():
        with open(jf) as f: r = json.load(f)
        ln_results.append(r)
        print(f"  [{i+1}/{len(LN_CFGS)}] D={D_val} s={seed} [SKIP] loss={r['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/{len(LN_CFGS)}] D={D_val} s={seed} ...")
    torch.manual_seed(seed)
    # FIX: D_val is model WIDTH (base_ch), not dataset size — use full CIFAR-10
    model = ThermoNet(base_ch=D_val, depth=3, norm='layernorm', skip=False)
    tr, va = loaders()
    # Full dataset (no Subset) — width scaling study
    r = train(model, tr, va, 200)
    r.update({"D": D_val, "seed": seed, "J_topo": J_LN, "norm": "layernorm"})
    ln_results.append(r)
    with open(jf, "w") as f: json.dump(r, f)
    print(f"    -> loss={r['best_val_loss']:.4f} acc={r['best_val_acc']:.4f} ep={r['best_epoch']}")
ln_time = time.time() - t0
print(f"\nLN done in {ln_time/60:.1f} min")


In [ ]:

# Fit LN scaling law
losses_by_d = {}
for r in ln_results:
    D = r["D"]
    losses_by_d.setdefault(D, []).append(r["best_val_loss"])
avg = {D: np.mean(v) for D, v in losses_by_d.items()}
Ds = np.array(list(avg.keys()))
Ls = np.array([avg[d] for d in Ds])

def plaw(D, a, b, c): return a * D**(-b) + c
try:
    popt, _ = curve_fit(plaw, Ds, Ls, p0=[10, 0.5, 0.5],
                         bounds=([0,0,0],[1000,3,3]), maxfev=10000)
    beta_LN = popt[1]
    r2 = 1 - (((Ls - plaw(Ds,*popt))**2).sum() / ((Ls-Ls.mean())**2).sum())
    print(f"LN: beta={beta_LN:.3f} R2={r2:.4f}")
    print(f"Compare: None beta=1.117, BN beta=0.950")
except Exception as e:
    beta_LN = None
    print(f"Fit failed: {e}")



## ══ Part 2: Phase B — Stratified Sampling ══
**100 configs** (6 widths x 4 depths x 3 norms x 2 skips)
**Sampling:** stratified — equal norm x skip representation

In [ ]:

WIDTHS = [16, 24, 32, 48, 64, 96]
DEPTHS = [3, 4, 5, 6]
NORMS  = ['none', 'batchnorm', 'layernorm']
SKIPS  = [False, True]

all_cfgs = [{"width":w,"depth":d,"norm":n,"skip":s}
            for w in WIDTHS for d in DEPTHS for n in NORMS for s in SKIPS]
print(f"Total space: {len(all_cfgs)} configs")

# Stratified: 8 per (norm, skip) = 8*3*2 = 48 total... make it 8 each norm * 6 widths = 96
random.seed(42)
stratified = []
for n in NORMS:
    for s in SKIPS:
        pool = [c for c in all_cfgs if c["norm"]==n and c["skip"]==s]
        stratified.extend(random.sample(pool, min(8, len(pool))))
# Add more random to reach ~100
remaining = [c for c in all_cfgs if c not in stratified]
stratified.extend(random.sample(remaining, 100 - len(stratified)))
CANDIDATES = stratified
print(f"Stratified sample: {len(CANDIDATES)}")
n_none = sum(1 for c in CANDIDATES if c["norm"]=="none")
n_bn   = sum(1 for c in CANDIDATES if c["norm"]=="batchnorm")
n_ln   = sum(1 for c in CANDIDATES if c["norm"]=="layernorm")
print(f"  none={n_none}, bn={n_bn}, ln={n_ln}")



In [ ]:

# J_topo for all candidates (LN NOT zeroed)
print("Computing J_topo for all candidates...")
t0 = time.time()
for c in CANDIDATES:
    c["J_topo"] = get_J_topo(c["width"], c["depth"], c["norm"], c["skip"])
print(f"Done in {time.time()-t0:.1f}s")
J_vals = [c["J_topo"] for c in CANDIDATES]
print(f"J_topo: [{min(J_vals):.4f}, {max(J_vals):.4f}], mean={np.mean(J_vals):.4f}")



## ══ Stage 1: L1 Screening (10 epochs) ══

In [ ]:

# Random arm: 30 configs x 10 epochs
random.seed(12345)
RAND30 = random.sample(CANDIDATES, 30)
RAND_RESULTS = []
t0 = time.time()
for i, c in enumerate(RAND30):
    jf = OUT / f"rand_l1_{i}.json"
    if jf.exists():
        with open(jf) as f: r = json.load(f)
        RAND_RESULTS.append(r)
        print(f"  [{i+1}/30] [SKIP] loss={r['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/30] {c['width']}/{c['depth']}/{c['norm']}/{c['skip']} ...")
    torch.manual_seed(42+i)
    model = ThermoNet(base_ch=c["width"], depth=c["depth"], norm=c["norm"], skip=c["skip"])
    tr, va = loaders()
    r = train(model, tr, va, 10)
    r["config"] = c
    RAND_RESULTS.append(r)
    with open(jf, "w") as f: json.dump(r, f)
    print(f"    -> loss={r['best_val_loss']:.4f}")
print(f"
Random L1 done in {(time.time()-t0)/60:.1f} min")



In [ ]:

# HBO arm: J_topo top-30 x 10 epochs
HBO30 = sorted(CANDIDATES, key=lambda x: x["J_topo"], reverse=True)[:30]
print(f"HBO J_topo range: [{HBO30[-1]['J_topo']:.4f}, {HBO30[0]['J_topo']:.4f}]")
HBO_RESULTS = []
t0 = time.time()
for i, c in enumerate(HBO30):
    orig_idx = CANDIDATES.index(c)
    jf = OUT / f"hbo_l1_{orig_idx}.json"
    if jf.exists():
        with open(jf) as f: r = json.load(f)
        HBO_RESULTS.append(r)
        print(f"  [{i+1}/30] idx={orig_idx} [SKIP] loss={r['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/30] idx={orig_idx} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']} ...")
    torch.manual_seed(42+orig_idx+1000)
    model = ThermoNet(base_ch=c["width"], depth=c["depth"], norm=c["norm"], skip=c["skip"])
    tr, va = loaders()
    r = train(model, tr, va, 10)
    r["config"] = c
    HBO_RESULTS.append(r)
    with open(jf, "w") as f: json.dump(r, f)
    print(f"    -> loss={r['best_val_loss']:.4f}")
print(f"
HBO L1 done in {(time.time()-t0)/60:.1f} min")



## ══ Stage 2: Selection (L1 loss based) ══

**Note:** GP removed to avoid data leakage. J_topo screening utility is tested by:
- HBO arm: select top-30 by J_topo → select top-5 by L1 loss
- Random arm: select top-5 by L1 loss directly
- Compare: does J_topo pre-screening find better final configs than random?


In [ ]:
# Select top-5 from each arm by L1 loss (no GP — avoids data leakage)
RAND_TOP5 = sorted(RAND_RESULTS, key=lambda x: x["best_val_loss"])[:5]
HBO_TOP5  = sorted(HBO_RESULTS,  key=lambda x: x["best_val_loss"])[:5]

print("Random top-5 (by L1 loss):")
for r in RAND_TOP5:
    c = r["config"]
    print(f"  L1={r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")

print("\nHBO top-5 (J_topo pre-screen, then L1):")
for r in HBO_TOP5:
    c = r["config"]
    print(f"  L1={r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")


In [ ]:
# J_topo quality check: does HBO arm have higher J_topo on average?
J_rand = np.array([r["config"]["J_topo"] for r in RAND_RESULTS])
J_hbo  = np.array([r["config"]["J_topo"] for r in HBO_RESULTS])
print(f"\nJ_topo: Random mean={J_rand.mean():.4f}, HBO mean={J_hbo.mean():.4f}")
print(f"HBO J_topo advantage: {J_hbo.mean() - J_rand.mean():.4f}")


## ══ Stage 2: L2 Training (top-5 x 50 epochs) ══

In [ ]:

# Random top-5 x 50 epochs
print("--- Random top-5 x 50 epochs ---")
RAND_L2 = []
for i, r in enumerate(RAND_TOP5):
    cfg = r["config"]
    orig_idx = CANDIDATES.index(cfg)
    jf = OUT / f"rand_l2_{orig_idx}.json"
    if jf.exists():
        with open(jf) as f: r2 = json.load(f)
        RAND_L2.append(r2)
        print(f"  [{i+1}/5] [SKIP] loss={r2['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/5] ...")
    torch.manual_seed(42+orig_idx)
    model = ThermoNet(base_ch=cfg["width"], depth=cfg["depth"], norm=cfg["norm"], skip=cfg["skip"])
    tr, va = loaders()
    r2 = train(model, tr, va, 50)
    r2["config"] = cfg
    RAND_L2.append(r2)
    with open(jf, "w") as f: json.dump(r2, f)
    print(f"    -> loss={r2['best_val_loss']:.4f} acc={r2['best_val_acc']:.4f}")



In [ ]:

# HBO top-5 x 50 epochs
print("--- HBO top-5 x 50 epochs ---")
HBO_L2 = []
for i, r in enumerate(HBO_TOP5):
    cfg = r["config"]
    orig_idx = CANDIDATES.index(cfg)
    jf = OUT / f"hbo_l2_{orig_idx}.json"
    if jf.exists():
        with open(jf) as f: r2 = json.load(f)
        HBO_L2.append(r2)
        print(f"  [{i+1}/5] [SKIP] loss={r2['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/5] ...")
    torch.manual_seed(42+orig_idx+1000)
    model = ThermoNet(base_ch=cfg["width"], depth=cfg["depth"], norm=cfg["norm"], skip=cfg["skip"])
    tr, va = loaders()
    r2 = train(model, tr, va, 50)
    r2["config"] = cfg
    HBO_L2.append(r2)
    with open(jf, "w") as f: json.dump(r2, f)
    print(f"    -> loss={r2['best_val_loss']:.4f} acc={r2['best_val_acc']:.4f}")



print("="*60)
print("FINAL RESULTS: J_topo Screening Efficiency")
print("="*60)
rl2 = [r["best_val_loss"] for r in RAND_L2]
hl2 = [r["best_val_loss"] for r in HBO_L2]

print('\n--- 50-epoch best val_loss ---')
print(f"Random: best={min(rl2):.4f}, mean={np.mean(rl2):.4f}")
print(f"HBO:    best={min(hl2):.4f}, mean={np.mean(hl2):.4f}")
delta = min(rl2) - min(hl2)
print(f"Improvement: HBO best {delta:.4f} lower than Random")

print('\n--- Compute efficiency ---')
print("Both arms: 30x10ep + 5x50ep = same compute budget")
print("HBO advantage: J_topo pre-screening costs ZERO extra compute")

print('\nRandom L2 top-5:')
for r in sorted(RAND_L2, key=lambda x: x["best_val_loss"]):
    c = r["config"]
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")
print('\nHBO L2 top-5:')
for r in sorted(HBO_L2, key=lambda x: x["best_val_loss"]):
    c = r["config"]
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")

rho, p = spearmanr([r['best_val_loss'] for r in RAND_L2],
                   [r['best_val_loss'] for r in RAND_TOP5])
print(f'\nL1->L2 Spearman: r={rho:.3f} p={p:.4f}")

J_rand = np.array([r["config"]["J_topo"] for r in RAND_RESULTS])
J_hbo  = np.array([r["config"]["J_topo"] for r in HBO_RESULTS])
print(f'\nJ_topo: Random mean={J_rand.mean():.4f}, HBO mean={J_hbo.mean():.4f}")

print('\n--- Interpretation ---')
if delta < -0.01:
    print("POSITIVE: J_topo pre-screening finds meaningfully better architectures.")
elif delta < 0.01:
    print("NEUTRAL: J_topo pre-screening offers no advantage at this fidelity.")
else:
    print("NEGATIVE: Random selection outperforms J_topo pre-screening.")
print("Note: This tests PRACTICAL utility, not asymptotic J_topo->E_floor theory.")


In [ ]:

print("="*60)
print("FINAL RESULTS: HBO vs Random (50-epoch)")
print("="*60)
rl2 = [r["best_val_loss"] for r in RAND_L2]
hl2 = [r["best_val_loss"] for r in HBO_L2]
print(f"
Random best={min(rl2):.4f}, mean={np.mean(rl2):.4f}")
print(f"HBO    best={min(hl2):.4f}, mean={np.mean(hl2):.4f}")
print(f"Delta (Random-HBO): {min(rl2)-min(hl2):.4f}")
print("
Random L2 top-5:")
for r in sorted(RAND_L2, key=lambda x: x["best_val_loss"]):
    c = r["config"]
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")
print("
HBO L2 top-5:")
for r in sorted(HBO_L2, key=lambda x: x["best_val_loss"]):
    c = r["config"]
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")
# L1->L2 correlation
rho, p = spearmanr([r['best_val_loss'] for r in RAND_L2],
                   [r['best_val_loss'] for r in RAND_TOP5])
print(f"
L1->L2 Spearman: r={rho:.3f} p={p:.4f}")



## Save + Push

In [ ]:

final = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "beta_LN": float(beta_LN) if beta_LN else None,
    "random_l2": [{"loss":r["best_val_loss"],"acc":r["best_val_acc"],"cfg":r["config"]} for r in RAND_L2],
    "hbo_l2":    [{"loss":r["best_val_loss"],"acc":r["best_val_acc"],"cfg":r["config"]} for r in HBO_L2],
    "random_best": float(min(rl2)),
    "hbo_best":     float(min(hl2)),
    "delta":        float(min(rl2)-min(hl2)),
}
with open(OUT/"phase_b_fixed_results.json","w") as f:
    json.dump(final, f, indent=2)
print(f"Results: {OUT}/phase_b_fixed_results.json")
for cmd in [
    f"git add {OUT}/*.json 2>/dev/null || true",
    f"git add {CKPT}/*.pt 2>/dev/null || true",
    "git add experiments/phase_b/notebooks/*.ipynb 2>/dev/null || true",
    "git commit -m 'Data: Phase B fixed HBO vs Random' 2>/dev/null || true",
    "git push origin develop 2>&1 || true",
]:
    get_ipython().system(cmd)
print("Done")

